# 📖 Notebook 1: RPO & RTO Fundamentals

## Why This Matters

Imagine you run an online store. Your database crashes at 2:00 PM.
Your last backup was at midnight. That means you just **lost 14 hours of
orders, payments, and customer data**.

How much money did you lose? How many customers are furious?

This is exactly why enterprises like Microsoft, JPMorgan, and hospitals
spend millions on **Business Continuity and Disaster Recovery (BCDR)**.

In this notebook, you will learn the two most important numbers in BCDR:
- **RPO** (Recovery Point Objective) — how much data you can afford to lose
- **RTO** (Recovery Time Objective) — how long you can afford to be down

## Learning Objectives

By the end of this notebook, you will understand:
- The difference between RPO and RTO
- How to calculate the business impact of downtime
- SLA math (what "five nines" really means)
- How different standby types affect RPO and RTO
- How to choose the right strategy for your system

## 🛠️ Setup

Start the infrastructure first:

```bash
cd enterprise-patterns/bcdr
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it does not appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import time
import datetime
from tabulate import tabulate

# Database connection settings
DB_PRIMARY = {
    "host": "localhost",
    "port": 5432,
    "database": "bcdr_demo",
    "user": "demo",
    "password": "demo"
}

DB_STANDBY = {
    "host": "localhost",
    "port": 5433,
    "database": "bcdr_demo",
    "user": "demo",
    "password": "demo"
}

def get_primary_connection():
    return psycopg2.connect(**DB_PRIMARY)

def get_standby_connection():
    return psycopg2.connect(**DB_STANDBY)

# Test connection
try:
    conn = get_primary_connection()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM orders")
    count = cur.fetchone()[0]
    print(f"✅ Connected to primary. Found {count} orders in the database.")
    conn.close()
except Exception as e:
    print(f"❌ Could not connect to primary: {e}")
    print("Make sure docker-compose is running: docker-compose up -d")

✅ Connected to primary. Found 500 orders in the database.


## 📚 What is RPO? (Recovery Point Objective)

**RPO answers: "How much data can we afford to lose?"**

Think of RPO as looking **backward in time** from the disaster:

```
  ──────[last backup]──────────[new data written]──────[💥 DISASTER]──────
                        ▲                                    ▲
                        │◄──── This data is LOST ──────────►│
                        │         (this is RPO)              │
```

### Real-World RPO Examples

| System | RPO | Why |
|--------|-----|-----|
| Stock Exchange | ~0 seconds | Every trade is legally binding |
| Banking | < 1 second | Financial regulations require it |
| Hospital Records | < 1 minute | Patient safety depends on it |
| E-commerce | < 5 minutes | Losing orders = losing revenue |
| Blog | < 24 hours | Content can be rewritten |
| Dev/Test | Days | No real users affected |

### Key Insight
**Lower RPO = more expensive.** Near-zero RPO requires synchronous replication, which adds latency to every write.

In [2]:
# =============================================================================
# Demo: Visualizing Data Loss at Different RPO Levels
# =============================================================================

conn = get_primary_connection()
cur = conn.cursor()

cur.execute(
    "SELECT COUNT(*) as total_orders, "
    "SUM(total_amount) as total_revenue, "
    "AVG(total_amount) as avg_order_value "
    "FROM orders"
)
total_orders, total_revenue, avg_order = cur.fetchone()
conn.close()

# Simulate order rate: assume our 500 orders happened over 30 days
orders_per_hour = total_orders / (30 * 24)
revenue_per_hour = float(total_revenue) / (30 * 24)

print("=" * 65)
print("DATA LOSS ANALYSIS AT DIFFERENT RPO LEVELS")
print("=" * 65)
print(f"\nSystem stats: {total_orders} orders, ${total_revenue:,.2f} total revenue")
print(f"Average rate: {orders_per_hour:.1f} orders/hour, ${revenue_per_hour:,.2f}/hour\n")

rpo_scenarios = [
    ("24 hours (daily backup)",   24),
    ("1 hour (hourly backup)",     1),
    ("5 minutes (streaming rep.)", 5/60),
    ("0 seconds (sync rep.)",      0),
]

table = []
for label, hours in rpo_scenarios:
    lost_orders = int(orders_per_hour * hours)
    lost_revenue = revenue_per_hour * hours
    table.append([label, f"{lost_orders}", f"${lost_revenue:,.2f}"])

print(tabulate(table, headers=["RPO Level", "Orders Lost", "Revenue Lost"], tablefmt="grid"))
print("\n💡 Notice how daily backups could mean losing hundreds of orders!")

DATA LOSS ANALYSIS AT DIFFERENT RPO LEVELS

System stats: 500 orders, $274,017.96 total revenue
Average rate: 0.7 orders/hour, $380.58/hour

+----------------------------+---------------+----------------+
| RPO Level                  |   Orders Lost | Revenue Lost   |
+============================+===============+================+
| 24 hours (daily backup)    |            16 | $9,133.93      |
+----------------------------+---------------+----------------+
| 1 hour (hourly backup)     |             0 | $380.58        |
+----------------------------+---------------+----------------+
| 5 minutes (streaming rep.) |             0 | $31.72         |
+----------------------------+---------------+----------------+
| 0 seconds (sync rep.)      |             0 | $0.00          |
+----------------------------+---------------+----------------+

💡 Notice how daily backups could mean losing hundreds of orders!


## 📚 What is RTO? (Recovery Time Objective)

**RTO answers: "How long can we be down?"**

Think of RTO as looking **forward in time** from the disaster:

```
  ──────[💥 DISASTER]──────────[recovery work]──────[✅ BACK ONLINE]──────
              ▲                                            ▲
              │◄──────── System is DOWN ──────────────────►│
              │              (this is RTO)                 │
```

### Real-World RTO Examples

| System | RTO | Why |
|--------|-----|-----|
| Payment Processing | < 30 seconds | Every second = lost transactions |
| Hospital Systems | < 5 minutes | Patient care cannot wait |
| Banking | < 15 minutes | Regulatory requirement |
| E-commerce | < 1 hour | Customer trust + revenue |
| Internal Tools | < 4 hours | Employees can do other work |
| Reporting | < 24 hours | Reports can wait |

### Key Insight
**Lower RTO = more expensive.** A 30-second RTO requires hot standby servers running 24/7, automatic failover, and extensive testing.

In [3]:
# =============================================================================
# Demo: Cost of Downtime Calculator
# =============================================================================

ORDERS_PER_HOUR = orders_per_hour
AVG_ORDER_VALUE = float(avg_order)
SUPPORT_STAFF_HOURLY = 50
SUPPORT_STAFF_COUNT = 5
SLA_PENALTY_PER_HOUR = 1000
REPUTATION_COST_PER_HOUR = 500

print("=" * 65)
print("COST OF DOWNTIME ANALYSIS")
print("=" * 65)

rto_scenarios = [
    ("30 seconds",  0.5/60),
    ("5 minutes",   5/60),
    ("1 hour",      1),
    ("4 hours",     4),
    ("24 hours",    24),
]

table = []
for label, hours in rto_scenarios:
    lost_revenue = ORDERS_PER_HOUR * AVG_ORDER_VALUE * hours
    staff_cost = SUPPORT_STAFF_HOURLY * SUPPORT_STAFF_COUNT * max(hours, 1)
    sla_penalty = SLA_PENALTY_PER_HOUR * hours
    reputation = REPUTATION_COST_PER_HOUR * hours
    total = lost_revenue + staff_cost + sla_penalty + reputation
    table.append([label, f"${lost_revenue:,.0f}", f"${staff_cost:,.0f}",
                  f"${sla_penalty:,.0f}", f"${total:,.0f}"])

print(f"\nBusiness parameters:")
print(f"  Order rate: {ORDERS_PER_HOUR:.1f}/hour x ${AVG_ORDER_VALUE:,.2f} avg")
print(f"  Support: {SUPPORT_STAFF_COUNT} staff x ${SUPPORT_STAFF_HOURLY}/hour")
print(f"  SLA penalty: ${SLA_PENALTY_PER_HOUR}/hour\n")

print(tabulate(table,
    headers=["Downtime (RTO)", "Lost Revenue", "Staff Cost", "SLA Penalty", "TOTAL Cost"],
    tablefmt="grid"))
print("\n💡 This is why enterprises invest millions in reducing RTO!")

COST OF DOWNTIME ANALYSIS

Business parameters:
  Order rate: 0.7/hour x $548.04 avg
  Support: 5 staff x $50/hour
  SLA penalty: $1000/hour

+------------------+----------------+--------------+---------------+--------------+
| Downtime (RTO)   | Lost Revenue   | Staff Cost   | SLA Penalty   | TOTAL Cost   |
+==================+================+==============+===============+==============+
| 30 seconds       | $3             | $250         | $8            | $266         |
+------------------+----------------+--------------+---------------+--------------+
| 5 minutes        | $32            | $250         | $83           | $407         |
+------------------+----------------+--------------+---------------+--------------+
| 1 hour           | $381           | $250         | $1,000        | $2,131       |
+------------------+----------------+--------------+---------------+--------------+
| 4 hours          | $1,522         | $1,000       | $4,000        | $8,522       |
+-----------------

## 📚 SLA Math — What "Five Nines" Really Means

You will often hear availability described as "nines":

| SLA | Uptime % | Downtime per Year | Downtime per Month |
|-----|----------|-------------------|-------------------|
| Two nines | 99% | 3.65 days | 7.3 hours |
| Three nines | 99.9% | 8.76 hours | 43.8 minutes |
| Four nines | 99.99% | 52.6 minutes | 4.38 minutes |
| Five nines | 99.999% | 5.26 minutes | 26.3 seconds |

Each additional "nine" is exponentially harder and more expensive to achieve.

In [4]:
# =============================================================================
# Demo: SLA Calculator
# =============================================================================

print("=" * 65)
print("SLA AVAILABILITY CALCULATOR")
print("=" * 65)

sla_levels = [
    ("Two nines",   0.99),
    ("Three nines", 0.999),
    ("Four nines",  0.9999),
    ("Five nines",  0.99999),
    ("Six nines",   0.999999),
]

MINUTES_PER_YEAR = 365.25 * 24 * 60
MINUTES_PER_MONTH = 30.44 * 24 * 60

table = []
for name, avail in sla_levels:
    down_yr = MINUTES_PER_YEAR * (1 - avail)
    down_mo = MINUTES_PER_MONTH * (1 - avail)

    if down_yr >= 1440:
        yr_str = f"{down_yr/1440:.1f} days"
    elif down_yr >= 60:
        yr_str = f"{down_yr/60:.1f} hours"
    else:
        yr_str = f"{down_yr:.1f} min"

    if down_mo >= 60:
        mo_str = f"{down_mo/60:.1f} hours"
    elif down_mo >= 1:
        mo_str = f"{down_mo:.1f} min"
    else:
        mo_str = f"{down_mo*60:.1f} sec"

    table.append([name, f"{avail*100:.4f}%", yr_str, mo_str])

print()
print(tabulate(table,
    headers=["SLA Level", "Availability", "Downtime/Year", "Downtime/Month"],
    tablefmt="grid"))
print()
print("💡 Going from 99.9% to 99.99% = allowed downtime drops by 10x!")
print("   Requires hot standby, automatic failover, and multi-region.")

SLA AVAILABILITY CALCULATOR

+-------------+----------------+-----------------+------------------+
| SLA Level   | Availability   | Downtime/Year   | Downtime/Month   |
+=============+================+=================+==================+
| Two nines   | 99.0000%       | 3.7 days        | 7.3 hours        |
+-------------+----------------+-----------------+------------------+
| Three nines | 99.9000%       | 8.8 hours       | 43.8 min         |
+-------------+----------------+-----------------+------------------+
| Four nines  | 99.9900%       | 52.6 min        | 4.4 min          |
+-------------+----------------+-----------------+------------------+
| Five nines  | 99.9990%       | 5.3 min         | 26.3 sec         |
+-------------+----------------+-----------------+------------------+
| Six nines   | 99.9999%       | 0.5 min         | 2.6 sec          |
+-------------+----------------+-----------------+------------------+

💡 Going from 99.9% to 99.99% = allowed downtime drops by 10x

## 📚 Hot, Warm, and Cold Standby

The standby type determines both your RPO and RTO:

### Hot Standby (what we use in this lab)
- Server is **running and receiving real-time data updates**
- Can serve read-only queries while standing by
- Failover takes **seconds to minutes**
- **Example**: PostgreSQL streaming replication, Redis replica

### Warm Standby
- Server is **running but data is synced periodically** (e.g., every hour)
- Needs to "catch up" before it can become primary
- Failover takes **minutes to hours**
- **Example**: Periodic database snapshots restored to standby

### Cold Standby
- Server **exists but is turned off** (or does not exist yet)
- Must be started, then restore from backup
- Failover takes **hours to days**
- **Example**: AWS AMI that you launch and restore a backup to

In [5]:
# =============================================================================
# Demo: Check Our Hot Standby Replication Status
# =============================================================================

conn = get_primary_connection()
cur = conn.cursor()

cur.execute(
    "SELECT client_addr, state, sent_lsn, write_lsn, "
    "flush_lsn, replay_lsn, sync_state "
    "FROM pg_stat_replication"
)
rows = cur.fetchall()
conn.close()

if rows:
    print("=" * 65)
    print("REPLICATION STATUS (from primary)")
    print("=" * 65)
    for row in rows:
        print(f"  Standby address:  {row[0]}")
        print(f"  State:            {row[1]}")
        print(f"  Sent LSN:         {row[2]}")
        print(f"  Written LSN:      {row[3]}")
        print(f"  Flushed LSN:      {row[4]}")
        print(f"  Replayed LSN:     {row[5]}")
        print(f"  Sync mode:        {row[6]}")
    print()
    print("💡 async = writes don't wait for standby. Better perf but RPO > 0.")
    print("   sync = RPO 0, but every write waits for standby confirmation.")
else:
    print("⚠️  No standby connected.")
    print("   Run: docker-compose up -d pg-standby")

REPLICATION STATUS (from primary)
  Standby address:  192.168.192.7
  State:            streaming
  Sent LSN:         0/4000060
  Written LSN:      0/4000060
  Flushed LSN:      0/4000060
  Replayed LSN:     0/4000060
  Sync mode:        async

💡 async = writes don't wait for standby. Better perf but RPO > 0.
   sync = RPO 0, but every write waits for standby confirmation.


In [6]:
# =============================================================================
# Demo: Measure Replication Lag in Real-Time
# =============================================================================

primary_conn = get_primary_connection()
primary_conn.autocommit = True
primary_cur = primary_conn.cursor()

standby_conn = get_standby_connection()
standby_cur = standby_conn.cursor()

print("=" * 65)
print("MEASURING REPLICATION LAG")
print("=" * 65)
print("Writing a test record to primary, timing standby visibility...\n")

marker = f"RPO_TEST_{int(time.time())}"
write_start = time.time()

primary_cur.execute(
    "INSERT INTO audit_log (table_name, record_id, action, changed_by) "
    "VALUES (%s, %s, %s, %s)",
    ('rpo_test', 0, 'INSERT', marker)
)
write_time = time.time() - write_start

# Poll standby until it sees the record
poll_start = time.time()
found = False
attempts = 0

while time.time() - poll_start < 10:
    attempts += 1
    try:
        standby_conn.rollback()  # new snapshot
        standby_cur.execute(
            "SELECT id FROM audit_log WHERE changed_by = %s", (marker,)
        )
        if standby_cur.fetchone():
            found = True
            break
    except Exception:
        pass
    time.sleep(0.01)

lag = time.time() - poll_start

if found:
    print(f"  ✅ Write to primary:    {write_time*1000:.1f} ms")
    print(f"  ✅ Visible on standby:  {lag*1000:.1f} ms")
    print(f"  ✅ Poll attempts:       {attempts}")
    print(f"\n  RPO for this write: ~{lag*1000:.0f} ms")
else:
    print(f"  ⚠️ Record not seen on standby after 10s")

# Cleanup
primary_cur.execute("DELETE FROM audit_log WHERE changed_by = %s", (marker,))
primary_conn.close()
standby_conn.close()

MEASURING REPLICATION LAG
Writing a test record to primary, timing standby visibility...

  ✅ Write to primary:    2.7 ms
  ✅ Visible on standby:  2.3 ms
  ✅ Poll attempts:       1

  RPO for this write: ~2 ms


## 📝 Summary

### What You Learned

1. **RPO** — How much data you can lose. Lower RPO = faster replication = more cost.
2. **RTO** — How long you can be down. Lower RTO = hot standby + automation = more cost.
3. **SLA Math** — Each nine is 10x harder. Five nines = 5.26 min downtime/year.
4. **Standby Types** — Hot (seconds), Warm (minutes), Cold (hours).
5. **Business Impact** — Lost revenue + SLA penalties + staff costs + reputation.

### Key Takeaway

> **BCDR is not about preventing failures — failures WILL happen.**
> **BCDR is about how quickly and completely you recover.**

### Next Notebook

In **Notebook 2**, we dive into database replication — how PostgreSQL
streaming replication works, and how to perform a failover.